# Construct Goal Model

Train phase-specific second-stage goal models. Validation uses walk-forward time-series splits; final models are fitted on all matches available before the selected FIFA 2026 phase.

In [ ]:
import os

phase = 'round_of_16'
os.environ['WORLD_CUP_PHASE'] = phase

print(f"Set WORLD_CUP_PHASE to '{phase}'")

In [ ]:
from pathlib import Path

import pandas as pd

from utils.goal_model import (
    configure_goal_phase,
    evaluate_second_stage_on_historical_matches,
    get_goal_model_paths,
    summarize_second_stage_validation,
    train_and_save_goal_models,
)

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)

phase = 'round_of_16'
n_splits = 5
max_matches = None
min_history_matches = 1000

configure_goal_phase(phase)

In [ ]:
goal_total_model, goal_diff_model = train_and_save_goal_models(phase=phase)
goal_total_model_path, goal_diff_model_path = get_goal_model_paths(phase)

df_saved_models = pd.DataFrame([
    {
        'model': 'total_goals',
        'path': str(Path(goal_total_model_path)),
        'exists': Path(goal_total_model_path).exists(),
    },
    {
        'model': 'goal_diff',
        'path': str(Path(goal_diff_model_path)),
        'exists': Path(goal_diff_model_path).exists(),
    },
])

df_saved_models

In [ ]:
df_validation = evaluate_second_stage_on_historical_matches(
    phase=phase,
    n_splits=n_splits,
    max_matches=max_matches,
    min_history_matches=min_history_matches,
)
df_summary = summarize_second_stage_validation(df_validation)

print(f'Matches evaluated: {len(df_validation):,}')
df_summary

In [ ]:
df_result_breakdown = (
    df_validation.groupby('actual_result')
    .agg(
        matches=('actual_result', 'size'),
        home_goals_mae=('goal_error_a', 'mean'),
        away_goals_mae=('goal_error_b', 'mean'),
        total_goals_mae=('total_goal_error', 'mean'),
        goal_diff_mae=('goal_diff_error', 'mean'),
        exact_score_top1_accuracy=('exact_score_hit_at_top1', 'mean'),
        mean_actual_score_probability=('actual_score_probability', 'mean'),
    )
    .reset_index()
)

df_result_breakdown

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


df_plot = df_validation.copy()
df_plot['date'] = pd.to_datetime(df_plot['date'])
df_plot = df_plot.sort_values('date')

df_plot['total_goals_mae_rolling'] = (
    df_plot['total_goal_error']
    .rolling(window=50, min_periods=10)
    .mean()
)

df_plot['goal_diff_mae_rolling'] = (
    df_plot['goal_diff_error']
    .rolling(window=50, min_periods=10)
    .mean()
)

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(
    df_plot['date'],
    df_plot['total_goals_mae_rolling'],
    label='Total goals MAE, rolling 50 matches',
    linewidth=2,
)

ax.plot(
    df_plot['date'],
    df_plot['goal_diff_mae_rolling'],
    label='Goal diff MAE, rolling 50 matches',
    linewidth=2,
)

ax.set_title(f'Goal model validation performance ({phase})')
ax.set_xlabel('Match date')
ax.set_ylabel('Mean absolute error')
ax.legend()
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()